# 03 pycavi_run

This is a wrapper script to the mefpp engine to reduce the data that has been partioned and associated with the scripts

*pymefpp_partition_calibration
*pymefpp_partition_science
*pymefpp_associate

After running those scripts we end up with a  directory named reduced that contains the following tree structure:

* reduced/calibs
        * /bpm
        * /bias
        * /dome
        * /sky
    * /science
    
Below science we have all the science partition to be reduced.

This script will:

* take a science partition at a time
* execute the reduction cascade

The idea is to identify all biases, dome flats, sky flats that are needed for the full reduction of that science partitions, and only these, and reduce them. This permits a piecemeal approach to the work.


        
 

In [1]:
from os import listdir
from os.path import isfile, join
import glob
import logging

from astropy.io import fits
import math
import operator
import copy
import os
import shutil
from datetime import datetime
from pprint import pprint

# here we import the local mefpp_run library
import sys
sys.path.insert(0, os.getcwd())
sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_associate_reduction as mar
from pycavi import pycavi_tools
from pycavi import pycavi_instrument


This will be executed when the package is imported.


In [2]:
# where to find mefpp module
mefpp_dir = '/home/lceledon/Software/mefpp'

# Reading configuration file
#
# Asume that all notebooks are in a directory where the data reduction will
# be carried out from. It will contain initially all python libraries and 
# a the config subdirectory.
#
# debug_mode: True
# log_level: info
# observatory: Paranal
# telescope: VST
# instrument: OmegaCam
# file_prefix: OMEGA
# raw_data_dir: /Users/fselman/Pictures/OCAM/CPPup/raw/
# project_dir: /Users/fselman/Pictures/OCAM/CPPup2/
# reduced_calib_dir: /Users/fselman/Pictures/OCAM/CPPup2/calib/
# reduced_science_dir: /Users/fselman/Pictures/OCAM/CPPup2/science/
# project_logs_dir: /Users/fselman/Pictures/OCAM/CPPup2/logs/
# sex_dir_name: Sextractor_dir
# scamp_dir_name: Scamp_dir
# swarp_dir_name: Swarp_dir
# cal_dir_name: Calibration_dir
#
config_dir = os.getcwd() + '/' + '../config/'
config = pycavi_tools.read_pycavi_config(config_dir)
instrument = config['Provenance']['instrument'].lower()
raw_data_dir = config['Directories']['raw_data_dir']
project_dir = config['Directories']['project_dir']
static_cal_dir = config['Directories']['static_calib_dir']
reduced_cal_dir = config['Directories']['reduced_calib_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['project_dir']     # Note that for this notebook we set the reduced science dir to the root project dir.
log_dir = config['Directories']['project_logs_dir']

log_file_run = log_dir + "/pymeffpp_run.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_run),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# Should read the instrument configuration file
instrument = pycavi_instrument.Instrument(config_dir=config_dir, instrument=instrument)

os.chdir(project_dir)
global cwd
cwd = os.getcwd()
print('cwd: ', cwd)


config_dir: /home/lceledon/Software/pycavi/Pycavi-main/Pycavi-main_LC/notebooks/../config/ instrument: omegacam
cwd:  /home/lceledon/Investigacion/OmegaCam/Work_dir


In [3]:
# seraching the disk for science and standard field partitions
# the search is recursive for *.list files. No more inteligence
# than that.
#
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))

std_list = glob.glob(reduced_std_dir+'**/std.list', recursive=True)
print('number of standard star partitions: ', len(std_list))

# associations for the science frames 
sci_partitions = []
for science_file in science_list:
    sci_partitions.append(mar.ExposuresPartition(science_file, 'SCIENCE', instrument))

# associations for the standard star field frames 
std_partitions = []
for std_file in std_list:
    std_partitions.append(mar.ExposuresPartition(std_file, 'STD', instrument))

# display of data partitions to select data to reduce
print('--------------------- science frame associations -------------------')
science_partition_list = sci_partitions+std_partitions
i = 0
for partition in science_partition_list:
    print(i, partition.base_path)
    i+=1

number of science partitions:  2
number of standard star partitions:  2
--------------------- science frame associations -------------------
0 /home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/NB_659/2017-03-13T01:57:49
1 /home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/r_SDSS/2017-03-13T01:57:49
2 /home/lceledon/Investigacion/OmegaCam/Work_dir//std/SA98/NB_659/2017-03-13T00:18:38
3 /home/lceledon/Investigacion/OmegaCam/Work_dir//std/SA98/r_SDSS/2017-03-12T23:45:37


In [4]:

# Selection and association was done in notebook 01.
# Here we rebuild the association in memory and reduce.
#
# Note that the loop is organize to reduce a science partition
# and its requirements at a time. For some projects the user
# might prefer to reduce all calibrations first, and then the science.
# For example, to check the quality of the calibrations before
# proceeding with the science.
#
# For testing you might want to select a single partition here
# example: 
#
# For that, select the index from the list displayed in the previous
# cell and add it to the idx_selected list. If this list is empty, then
# all science partitions will be reduced, one at a time.
#

idx_selected = [0,1]
for i in range(len(science_partition_list)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue

    scipart = science_partition_list[i]

    bias_part_list = []
    dome_part_list = []
    sky_part_list = []
    
    sci_part_list = []

    # -------------- DATA ASSOCIATION SECTION ---------------
    print('\n\n#---------------------- DATA ASSOCIATION TREE -------------------------')
    print('-------------------------------------------------------------------------')
    print('scipart: '+scipart.base_path)
    sci_part_list.append(scipart)
    
    mar.associateBPM(scipart, scipart, instrument)
    print('        bpm part:  '+scipart.bpmPartition.base_path)
    mar.associateBias(scipart, scipart, instrument)
    print('        bias part:  '+scipart.biasPartition.base_path)
    bias_part_list.append(scipart.biasPartition)
    mar.associateMFF(scipart, scipart, instrument)
    print('        MFF  part:  '+scipart.mffPartition.base_path)

    # associations for the MFF frame
    mffpart = scipart.mffPartition
    print('         mffpart: '+mffpart.base_path)
    mar.associateBPM(mffpart, scipart, instrument)
    print('                  bpm  part:  '+mffpart.bpmPartition.base_path)
    mar.associateBias(mffpart, scipart, instrument)
    print('                  bias part:  '+mffpart.biasPartition.base_path)
    bias_part_list.append(mffpart.biasPartition)
    mar.associateDome(mffpart, scipart, instrument)
    print('                  dome part:  '+mffpart.domePartition.base_path)
    dome_part_list.append(mffpart.domePartition)
    mar.associateSky(mffpart, scipart, instrument)
    print('                   sky part:  '+mffpart.skyPartition.base_path)
    sky_part_list.append(mffpart.skyPartition)

    # associations for the SKY frame
    skypart = mffpart.skyPartition
    print('  ---> skypart: '+skypart.base_path)
    mar.associateBPM(skypart, scipart, instrument)
    print('                  bpm  part:  '+skypart.bpmPartition.base_path)
    mar.associateBias(skypart, scipart, instrument)
    print('                  bias part:  '+skypart.biasPartition.base_path)
    bias_part_list.append(skypart.biasPartition)
    mar.associateDome(skypart, scipart, instrument)
    print('                  dome part:  '+skypart.domePartition.base_path)
    dome_part_list.append(skypart.domePartition)

    # associations for the dome frame
    domepart = mar.ExposuresPartition(skypart.domePartition.frame, 'FLAT,DOME', instrument)
    print('  ---> domepart: '+domepart.base_path)
    mar.associateBPM(domepart, scipart, instrument)
    print('                  bpm  part:  '+domepart.bpmPartition.base_path)
    mar.associateBias(domepart, scipart, instrument)
    print('                  bias part:  '+domepart.biasPartition.base_path)
    bias_part_list.append(domepart.biasPartition)
    print ('%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%')
    
    # --------------------- DATA REDUCTION SECTION ---------------------
    # here we reduce the bpm frames which for now is only a cp operation
    mar.mefpp_reduce_bpm(mefpp_dir, static_cal_dir, instrument)

    # we start reducing the biases
    mar.mefpp_reduce_bias(mefpp_dir, set(bias_part_list), instrument)

    # reduction of dome frames
    mar.mefpp_reduce_dome(mefpp_dir, set(dome_part_list), instrument)

    # reduction of sky frames
    mar.mefpp_reduce_sky(mefpp_dir, set(sky_part_list), instrument)
  
    # we reduce the science frames here
    mar.mefpp_reduce_science(mefpp_dir, set(sci_part_list), instrument)
    
    # finally we normalize the image to set the units in e-/s.
    # light correction is applied if the filter are u,g,r o i_sdss
    mar.mefpp_normalize_science(set(sci_part_list), instrument, apply_light_corr=False)

print("\n\n\n ------------------- DATA ASSOCIATION AND REDUCTION DONE ----------------------\n\n\n")



#---------------------- DATA ASSOCIATION TREE -------------------------
-------------------------------------------------------------------------
scipart: /home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/NB_659/2017-03-13T01:57:49
        bpm part:  /home/lceledon/Investigacion/OmegaCam/Work_dir/static_calib/bpm
        bias part:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-13T09:44:19
        MFF  part:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/sky/200253598/NB_659/2017-03-08T23:14:31
         mffpart: /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/sky/200253598/NB_659/2017-03-08T23:14:31
                  bpm  part:  /home/lceledon/Investigacion/OmegaCam/Work_dir/static_calib/bpm
                  bias part:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-09T09:40:35
                  dome part:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/dome/574184/NB_659/2017-03-09T09:40:35
         

2024-10-30 12:12:50,463:INFO:run_command: Running command: nohup /home/lceledon/Software/mefpp/bin/mefpp --bias bias.list  --biassave


mefpp_reduce_bpm: done copying BPMs

----------------------------- mefpp_reduce_bias ------------------------------
mefpp_reduce_bias: length of bias partition list: 4
Using paf: /home/lceledon/Software/pycavi/Pycavi-main/Pycavi-main_LC/notebooks/../config//omegacam.paf


basepath:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-09T09:40:35
bias_already_reduced:  False
bias_list_exists:  True
mefpp_paf_exists:  True
bpm_exists:  True
bias_list:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-09T09:40:35/bias.list


 /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-09T09:40:35 True True True False

 nohup /home/lceledon/Software/mefpp/bin/mefpp --bias bias.list  --biassave


2024-10-30 12:13:52,485:INFO:run_command: Running command: nohup /home/lceledon/Software/mefpp/bin/mefpp --bias bias.list  --biassave


success running nohup /home/lceledon/Software/mefpp/bin/mefpp --bias bias.list  --biassave


basepath:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-09T09:40:35
bias_already_reduced:  True
bias_list_exists:  True
mefpp_paf_exists:  True
bpm_exists:  True
bias_list:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-09T09:40:35/bias.list
Skipping bias reduction


basepath:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-09T09:40:35
bias_already_reduced:  True
bias_list_exists:  True
mefpp_paf_exists:  True
bpm_exists:  True
bias_list:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-09T09:40:35/bias.list
Skipping bias reduction
Using paf: /home/lceledon/Software/pycavi/Pycavi-main/Pycavi-main_LC/notebooks/../config//omegacam.paf


basepath:  /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/bias/574184/2017-03-13T09:44:19
bias_already_reduced:  False
bias_list_exists:  True
mefpp_paf

...

success running nohup /home/lceledon/Software/mefpp/bin/mefpp  --bias OCAM_MB_1x1.fits  --dome dome.list  --domesave


base_path: /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/dome/574184/r_SDSS/2017-03-14T09:41:34
dome_list_exists: True
paf exists: True
bpm exists: True
Master bias exists: True
dome already reduced: True
skipping dome reduction

----------------------------- mefpp_reduce_sky (and mff)  ------------------------------
mefpp_reduce_sky: length of sky partition list: 1
Using paf: /home/lceledon/Software/pycavi/Pycavi-main/Pycavi-main_LC/notebooks/../config//omegacam.paf


base_path: /home/lceledon/Investigacion/OmegaCam/Work_dir/calib/sky/200253584/r_SDSS/2017-03-13T23:12:13
paf exists: True
bpm exists: True
Master bias exists: True
domeff_exists: True
sky already reduced: False

 nohup /home/lceledon/Software/mefpp/bin/mefpp  --bias OCAM_MB_1x1.fits  --dome OCAM_DOMEFF_r_SDSS_1x1.fits  --sky sky.list  --skysave  --mffsave OCAM_MFF


2024-10-30 12:26:07,167:INFO:run_command: Running command: nohup /home/lceledon/Software/mefpp/bin/mefpp  --bias OCAM_MB_1x1.fits  --mff OCAM_MFF_r_SDSS_1x1.fits  --science science.list


success running nohup /home/lceledon/Software/mefpp/bin/mefpp  --bias OCAM_MB_1x1.fits  --dome OCAM_DOMEFF_r_SDSS_1x1.fits  --sky sky.list  --skysave  --mffsave OCAM_MFF

----------------------------- mefpp_reduce_science ------------------------------
mefpp_reduce_science: length of science partition list: 1
Using paf: /home/lceledon/Software/pycavi/Pycavi-main/Pycavi-main_LC/notebooks/../config//omegacam.paf
/home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/r_SDSS/2017-03-13T01:57:49 True True True True True


base_path: /home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/r_SDSS/2017-03-13T01:57:49
paf exists: True
bpm exists: True
Master bias exists: True
Master flat field exists: True
sci list exists: True

 nohup /home/lceledon/Software/mefpp/bin/mefpp  --bias OCAM_MB_1x1.fits  --mff OCAM_MFF_r_SDSS_1x1.fits  --science science.list
success running nohup /home/lceledon/Software/mefpp/bin/mefpp  --bias OCAM_MB_1x1.fits  --mff OCAM_MFF_r_SDSS_1x1.fits  --science

2024-10-30 12:28:08,104:INFO:mefpp_normalize_science: Setting AIRMASS to 1.051
2024-10-30 12:28:08,125:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:08,143:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:08,176:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:08,210:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:08,246:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:08,568:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:09,792:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:10,470:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:11,223:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:11,972:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:12,507:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:13,240:INFO:mefpp_normalize_science: NO light correction.
2024-10-30 12:28:14,2




 ------------------- DATA ASSOCIATION AND REDUCTION DONE ----------------------



